# Reproducibility smoke test on the free sample

This notebook runs `../validation/benchmark_script.py`, **unmodified and identical to the script used for the published benchmark**, against the free sample shipped in `../sample/`.

**Purpose:** show that the code runs end to end against the real delivered schema.

**Not the purpose:** reproduce the published metric. The sample holds 600 of the 105,000 subjects (100 per drug/route/arm group). The benchmark compares group medians with published ranges, and medians over 100 subjects are not the medians over the full groups, so a PASS or FAIL here can differ from the published result.

The number to cite is the one measured on the full dataset: 41 of 41 structural checks and 8 of 10 literature benchmarks passed, with two disclosed failures: scopolamine terminal half-life 1.89 h vs 2.80-6.20 h published, and scopolamine patch Cavg 209.78 pg/mL vs 52.20-139.20 pg/mL (FDA label about 87).


In [1]:
import sys, subprocess, shutil, tempfile, pathlib
import pandas as pd
HERE = pathlib.Path.cwd()
SAMPLE = (HERE / ".." / "sample").resolve()
SCRIPT = (HERE / ".." / "validation" / "benchmark_script.py").resolve()
work = pathlib.Path(tempfile.mkdtemp(prefix="smoke_"))   # all outputs go here, nothing is written into the repository
print("sample dir :", SAMPLE.name)
print("script     :", SCRIPT.name, "(unmodified copy of the published benchmark script)")
def clean(text):
    return text.replace(str(work), "<tmp>").replace(str(SAMPLE), "<sample>")


sample dir : sample
script     : benchmark_script.py (unmodified copy of the published benchmark script)


In [2]:
# The script reads fixed relative paths (DATASET/subjects.parquet and so on). The sample files carry a
# "_sample" suffix, so copy them under the names the script expects, inside a temporary directory.
(work / "DATASET").mkdir()
(work / "VALIDATION").mkdir()
for name in ["subjects", "truth_pk_parameters", "truth_pk_metrics"]:
    shutil.copy(SAMPLE / f"{name}_sample.parquet", work / "DATASET" / f"{name}.parquet")
shutil.copy(SCRIPT, work / "benchmark_script.py")
r = subprocess.run([sys.executable, "benchmark_script.py"], cwd=work, capture_output=True, text=True)
print(clean(r.stdout))
print(clean(r.stderr[-1500:]))
print("exit code:", r.returncode)
results = pd.read_parquet(work / "VALIDATION" / "benchmark_results.parquet")
results

                                                 metric          drug  dataset_value  published_range_lo  published_range_hi         status
                                    CL (median, GROUND) acetaminophen        17.5887             14.1950             24.1500           PASS
                          V1+V2 per kg (median, GROUND) acetaminophen         0.6280              0.3000              1.0000           PASS
                                  Tmax (median, GROUND) acetaminophen         1.3800              1.0500              1.9500           PASS
                              t_half_z (median, GROUND)      zolpidem         2.5979              1.5000              3.9000           PASS
                                  Tmax (median, GROUND)   scopolamine         2.0900              1.4000              2.6000           PASS
                              t_half_z (median, GROUND)   scopolamine         1.8945              2.8000              6.2000 FAIL_DISCLOSED
                    

,metric,drug,dataset_value,published_value_desc,published_range_lo,published_range_hi,citation,status
0,"CL (median, GROUND)",acetaminophen,17.5887,Allegaert 2014 16.7 L/h; DrugBank/FDA 0.27 L/h...,14.1950,24.1500,REF-08 (Allegaert 2014); REF-09 (DrugBank); RE...,PASS
1,"V1+V2 per kg (median, GROUND)",acetaminophen,0.6280,DrugBank/FDA ~0.9 L/kg; Guzmán 2016 0.95+-0.12...,0.3000,1.0000,REF-08 (Allegaert 2014); REF-09 (DrugBank); RE...,PASS
2,"Tmax (median, GROUND)",acetaminophen,1.3800,DrugBank/PharmGKB oral Tmax = 1.5 h (this prod...,1.0500,1.9500,REF-09 (DrugBank/PharmGKB),PASS
3,"t_half_z (median, GROUND)",zolpidem,2.5979,Cha 2024 CL/Vd implies 2.53 h; Salva & Costa 1...,1.5000,3.9000,REF-12 (Cha 2024); REF-13 (Salva & Costa 1995)...,PASS
4,"Tmax (median, GROUND)",scopolamine,2.0900,Cintron et al. 1987b control-arm Tmax = 2.0 h ...,1.4000,2.6000,REF-03 (Dello Russo 2022); REF-07 (NASA TM-58280),PASS
5,"t_half_z (median, GROUND)",scopolamine,1.8945,Putcha et al. 1989 report 4.5+-1.7 h directly ...,2.8000,6.2000,REF-11 (Putcha et al. 1989),FAIL_DISCLOSED
6,"Tmax (median, GROUND)",zolpidem,0.9200,Salva & Costa 1995 report 0.75-2.6 h; FDA TOVA...,0.7500,2.6000,REF-13 (Salva & Costa 1995); REF-14 (FDA label),PASS
7,"FLIGHT/GROUND Cmax ratio, short-duration class...",acetaminophen,0.8905,"Cintron et al. 1987a (short Shuttle missions, ...",0.5285,1.9026,REF-03/REF-07 (Cintron 1987a via Dello Russo 2...,PASS
8,"FLIGHT/GROUND Cmax ratio, long-duration class ...",acetaminophen,0.5828,"Kovachevich 2009 (tablets 4.80/5.13=0.936, cap...",0.2350,1.4040,"REF-03 (Dello Russo 2022, Kovachevich 2009 + P...",PASS


## Reading the output

The scopolamine patch benchmark is skipped with a warning: it needs the simulation kernel (`orbitpk_kernel`), which ships with the generator in the full delivery and is not part of the sample. The remaining nine comparisons run on the sample's ground-arm medians. The scopolamine half-life check fails here for the same reason it fails on the full data; that failure is disclosed and explained in the dataset description above. The executed run reports 8 PASS, 1 FAIL_DISCLOSED and 0 FAIL_UNEXPECTED over the nine comparisons it could compute.